In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

churn = pd.read_csv("telco-data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(f'telco_customer_churn: {churn.shape[0]:>7,} rows  x  {churn.shape[1]:>3} columns')

In [ ]:
y = churn["Churn"]
X = churn.drop(columns=["customerID", "Churn"])

print(y.value_counts())
print(f"\nX: {X.shape[0]:,} rows x {X.shape[1]} columns")
X.head()

### boolean cols

`Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`, `gender`: each only ever takes 2 values, so each becomes a single 0/1 column with no information loss.

In [ ]:
binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "gender"]

# confirm each really only has 2 unique values before mapping
for col in binary_cols:
    print(col, X[col].unique())

In [ ]:
binary_map = {
    "Partner": {"Yes": 1, "No": 0},
    "Dependents": {"Yes": 1, "No": 0},
    "PhoneService": {"Yes": 1, "No": 0},
    "PaperlessBilling": {"Yes": 1, "No": 0},
    "gender": {"Male": 1, "Female": 0},
}

for col, mapping in binary_map.items():
    X[col] = X[col].map(mapping)
    assert X[col].isna().sum() == 0, f"{col} has values outside the expected mapping"

X[binary_cols].head()

### nominal cols

`InternetService`, `PaymentMethod`, `MultipleLines`: 3+ categories with no ranking between them. One-hot encode so each category gets its own independent 0/1 flag instead of a false ordering on one number line.

In [ ]:
nominal_cols = ["InternetService", "PaymentMethod", "MultipleLines"]

# confirm the category values before encoding
for col in nominal_cols:
    print(col, X[col].unique())

In [ ]:
X = pd.get_dummies(X, columns=nominal_cols, drop_first=True, dtype=int)

print(f"X: {X.shape[0]:,} rows x {X.shape[1]} columns")
X.filter(regex="InternetService_|PaymentMethod_|MultipleLines_").head()

### flagging internet dependent columns

`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`: each has a `"No internet service"` value that's just a repeat of the feature `InternetService == "No"`.

In [ ]:
internet_dependent_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]

no_internet_mask = churn["InternetService"] == "No"

for col in internet_dependent_cols:
    print(col, churn[col].unique())
    # every "No internet service" row should line up exactly with InternetService == "No"
    matches = (churn[col] == "No internet service") == no_internet_mask
    print(f"  matches InternetService == 'No' for all rows: {matches.all()}\n")

turning  internet-dependent columns to boolean:

In [ ]:
for col in internet_dependent_cols:
    X[col] = X[col].replace("No internet service", "No")
    print(col, X[col].unique())

In [ ]:
for col in internet_dependent_cols:
    X[col] = X[col].map({"Yes": 1, "No": 0})
    assert X[col].isna().sum() == 0, f"{col} has values outside the expected mapping"

X[internet_dependent_cols].head()

### ordinal cols

`Contract`:  3 categories w/ order. month-to-month == 0, one year == 1, two year == 2

In [ ]:
print("Contract", X["Contract"].unique())

contract_order = ["Month-to-month", "One year", "Two year"]
contract_map = {category: rank for rank, category in enumerate(contract_order)}
print(contract_map)

In [ ]:
X["Contract"] = X["Contract"].map(contract_map)
assert X["Contract"].isna().sum() == 0, "Contract has values outside the expected mapping"

X["Contract"].value_counts().sort_index()

### numeric cols flag only

`tenure`, `MonthlyCharges`, `TotalCharges`: already numbers that mean what they look like, no transformation needed
`SeniorCitizen` is already 0/1, so it's boolean even though its dtype is numeric.

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]

print(X[numeric_cols].dtypes)
X[numeric_cols].describe()

In [ ]:
# flag only: TotalCharges looks numeric but check its actual dtype before assuming "no encoding needed"
non_numeric = pd.to_numeric(X["TotalCharges"], errors="coerce").isna()
print(f"TotalCharges dtype: {X['TotalCharges'].dtype}")
print(f"rows that fail to parse as numeric: {non_numeric.sum()}")
X.loc[non_numeric, ["tenure", "TotalCharges"]]

In [ ]:
X["TotalCharges"] = pd.to_numeric(X["TotalCharges"], errors="coerce")

# the 11 unparseable rows are all brand-new customers (tenure == 0), so 0 total charges is correct
assert (X.loc[X["TotalCharges"].isna(), "tenure"] == 0).all(), "found a NaN TotalCharges row with nonzero tenure"
X["TotalCharges"] = X["TotalCharges"].fillna(0)

assert X["TotalCharges"].isna().sum() == 0, "TotalCharges still has missing values"
print(f"TotalCharges dtype: {X['TotalCharges'].dtype}")
X["TotalCharges"].describe()

### New features referencing EDA

### tenure_group

EDA showed churn risk isn't linear across `tenure. Bucket into 0–12 / 13–24 / 25–48 / 49+, then ordinal-encode (0→3)

In [ ]:
tenure_bins = [-1, 12, 24, 48, np.inf]
tenure_group_order = ["0-12", "13-24", "25-48", "49+"]

X["tenure_group"] = pd.cut(X["tenure"], bins=tenure_bins, labels=tenure_group_order)
X["tenure_group"].value_counts().reindex(tenure_group_order)

In [ ]:
tenure_group_map = {group: rank for rank, group in enumerate(tenure_group_order)}
X["tenure_group"] = X["tenure_group"].map(tenure_group_map)

assert X["tenure_group"].isna().sum() == 0, "tenure_group has values outside the expected mapping"
print(tenure_group_map)
X["tenure_group"].value_counts().sort_index()

### avg_monthly_spend

`TotalCharges` is a combination of tenure length and monthly rate. Dividing `TotalCharges / tenure` to normalize. Subtituting `MonthlyCharges` for the 11 `tenure == 0` rows bc it means the customers are new

In [ ]:
zero_tenure_mask = X["tenure"] == 0

X["avg_monthly_spend"] = X["TotalCharges"] / X["tenure"].replace(0, np.nan)
X.loc[zero_tenure_mask, "avg_monthly_spend"] = X.loc[zero_tenure_mask, "MonthlyCharges"]

assert X["avg_monthly_spend"].isna().sum() == 0, "avg_monthly_spend still has missing values"
print(f"rows substituted with MonthlyCharges: {zero_tenure_mask.sum()}")
X.loc[zero_tenure_mask, ["tenure", "TotalCharges", "MonthlyCharges", "avg_monthly_spend"]].head()

### num_services

Row-wise sum of the seven already-binary service columns (`PhoneService` + the six internet add-ons) — a proxy for how invested/sticky a customer is, independent of any single add-on.

In [ ]:
service_cols = ["PhoneService"] + internet_dependent_cols

X["num_services"] = X[service_cols].sum(axis=1)

X["num_services"].value_counts().sort_index()

### checking multicollinearity

`tenure`, `MonthlyCharges`, `TotalCharges`, `avg_monthly_spend` all touch "how long/how much", checking for redundancy

In [ ]:
collinearity_cols = ["tenure", "MonthlyCharges", "TotalCharges", "avg_monthly_spend"]

corr_matrix = X[collinearity_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation matrix: tenure/charges cluster")
plt.tight_layout()
plt.show()

corr_matrix

Dropping `TotalCharges` bc it's basicallly `tenure` × `MonthlyCharges` so it's not giving any new information

In [ ]:
X = X.drop(columns=["TotalCharges"])

print(f"X: {X.shape[0]:,} rows x {X.shape[1]} columns")
X.head()

Dropping `avg_monthly_spend`bc of its correlation of 0.996 with `MonthlyCharges. since `avg_monthly_spend = TotalCharges / tenure` and `TotalCharges ≈ tenure × MonthlyCharges`, avg_monthly_spend is not giving any new info than MonthlyCharges

In [ ]:
X = X.drop(columns=["avg_monthly_spend"])

print(f"X: {X.shape[0]:,} rows x {X.shape[1]} columns")
X.head()